In [1]:
import os

os.cpu_count()

16

In [4]:
import subprocess

resultado = subprocess.run(
    [
        "python",
        "scalability-globalDT.py",
        "4",                  # cores
        "10",                 # percentage
        "10",                 # max_depth
        "speedup_globalDT.csv" # filename
    ],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)

26/08/11 10:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Building phase took: 7.613654613494873 seconds
7.613654613494873
CORRECTO: el proceso con PID 21756 (proceso secundario de PID 7248)
ha sido terminado.
CORRECTO: el proceso con PID 7248 (proceso secundario de PID 5864)
ha sido terminado.
CORRECTO: el proceso con PID 5864 (proceso secundario de PID 20844)
ha sido terminado.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 1:>                                                          (0 + 4) / 4]

[Stage 1:==============>                                            (1 + 3) / 4]

                                                                                

[Stage 7:>                                                          (0 + 4) / 4]

                                                                        

In [5]:
resultado

CompletedProcess(args=['python', 'scalability-globalDT.py', '4', '10', '10', 'speedup_globalDT.csv'], returncode=0, stdout='26/08/11 10:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nBuilding phase took: 7.613654613494873 seconds\n7.613654613494873\nCORRECTO: el proceso con PID 21756 (proceso secundario de PID 7248)\nha sido terminado.\nCORRECTO: el proceso con PID 7248 (proceso secundario de PID 5864)\nha sido terminado.\nCORRECTO: el proceso con PID 5864 (proceso secundario de PID 20844)\nha sido terminado.\n', stderr='Setting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n\n[Stage 1:>                                                          (0 + 4) / 4]\n\n[Stage 1:==============>                                            (1 + 3) / 4]\n\n                                                                                \n\n[S

In [1]:
import subprocess
import os

# ============================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================

# Profundidad máxima del árbol
max_depth = 10

# Número de repeticiones de cada experimento
num_repetitions = 1

# Número de cores que queremos probar
num_cores = [1, 2, 4, 8, 12]

# Tamaños del dataset que queremos utilizar
sizes = [10, 20, 40, 80, 100]

In [2]:
# ============================================================
# SPEED-UP
# ============================================================
#
# Para Speed-Up:
#
# - El tamaño del dataset se mantiene FIJO.
# - Cambiamos el número de cores.
#
# Aquí utilizamos el 100% del dataset.
#
# Experimentos:
#
#   (1 core, 100%)
#   (2 cores, 100%)
#   (4 cores, 100%)
#   (8 cores, 100%)
#   (12 cores, 100%)
#
# Cada experimento se repite 10 veces.
# ============================================================

speedup_file = f"speedup_global_{max_depth}.csv"

# Si existe un fichero anterior, lo eliminamos para no mezclar
# resultados de experimentos anteriores.
if os.path.exists(speedup_file):
    os.remove(speedup_file)

print("===== SPEED-UP =====")

for i in range(num_repetitions):

    for cores in num_cores:

        print(
            f"Repetición {i+1}/{num_repetitions} | "
            f"cores={cores} | datos=10%"
        )

        resultado = subprocess.run(
            [
                "python",
                "scalability-globalDT.py",
                str(cores),
                "10",
                str(max_depth),
                speedup_file
            ],
            capture_output=True,
            text=True
        )

        # Comprobamos si la ejecución ha terminado correctamente
        if resultado.returncode != 0:
            print("ERROR:")
            print(resultado.stderr)
            raise RuntimeError(
                f"Falló el experimento con {cores} cores"
            )


print("Speed-Up terminado.\n")

===== SPEED-UP =====
Repetición 1/1 | cores=1 | datos=100%
Repetición 1/1 | cores=2 | datos=100%
Repetición 1/1 | cores=4 | datos=100%
Repetición 1/1 | cores=8 | datos=100%
Repetición 1/1 | cores=12 | datos=100%
Speed-Up terminado.



In [3]:
# ============================================================
# SIZE-UP
# ============================================================
#
# Para Size-Up:
#
# - El número de cores se mantiene FIJO.
# - Aumentamos el tamaño del dataset.
#
# Utilizamos 4 cores para todos los experimentos.
#
# Experimentos:
#
#   (4 cores, 10%)
#   (4 cores, 20%)
#   (4 cores, 40%)
#   (4 cores, 80%)
#   (4 cores, 100%)
#
# Cada experimento se repite 10 veces.
# ============================================================

sizeup_file = f"sizeup_global_{max_depth}.csv"

if os.path.exists(sizeup_file):
    os.remove(sizeup_file)

print("===== SIZE-UP =====")

for i in range(num_repetitions):

    for percentage in sizes:

        print(
            f"Repetición {i+1}/{num_repetitions} | "
            f"cores=4 | datos={percentage}%"
        )

        resultado = subprocess.run(
            [
                "python",
                "scalability-globalDT.py",
                "4",
                str(percentage),
                str(max_depth),
                sizeup_file
            ],
            capture_output=True,
            text=True
        )

        if resultado.returncode != 0:
            print("ERROR:")
            print(resultado.stderr)
            raise RuntimeError(
                f"Falló el experimento con {percentage}% de datos"
            )


print("Size-Up terminado.\n")

===== SIZE-UP =====
Repetición 1/1 | cores=4 | datos=10%
Repetición 1/1 | cores=4 | datos=20%
Repetición 1/1 | cores=4 | datos=40%
Repetición 1/1 | cores=4 | datos=80%
Repetición 1/1 | cores=4 | datos=100%
Size-Up terminado.



In [4]:
# ============================================================
# SCALE-UP
# ============================================================
#
# Para Scale-Up aumentamos proporcionalmente:
#
#       número de cores
#
# y:
#
#       tamaño del dataset.
#
# Utilizamos las parejas:
#
#       1 core  -> 10%
#       2 cores -> 20%
#       4 cores -> 40%
#       8 cores -> 80%
#       12 cores -> 100%
#
# IMPORTANTE:
# En el script original de las diapositivas se utilizaba:
#
#       [1, 2, 4, 8, 10]
#
# porque los tamaños eran:
#
#       [10, 20, 40, 80, 100]
#
# Si queremos que la última ejecución utilice 12 cores,
# ya no existe una relación estrictamente proporcional.
#
# Por tanto, aquí utilizaremos 10 cores para el último punto.
# ============================================================

scaleup_cores = [1, 2, 4, 8, 10]
scaleup_sizes = [10, 20, 40, 80, 100]

scaleup_file = f"scaleup_global_{max_depth}.csv"

if os.path.exists(scaleup_file):
    os.remove(scaleup_file)

print("===== SCALE-UP =====")

for i in range(num_repetitions):

    for cores, percentage in zip(
        scaleup_cores,
        scaleup_sizes
    ):

        print(
            f"Repetición {i+1}/{num_repetitions} | "
            f"cores={cores} | datos={percentage}%"
        )

        resultado = subprocess.run(
            [
                "python",
                "scalability-globalDT.py",
                str(cores),
                str(percentage),
                str(max_depth),
                scaleup_file
            ],
            capture_output=True,
            text=True
        )

        if resultado.returncode != 0:
            print("ERROR:")
            print(resultado.stderr)
            raise RuntimeError(
                f"Falló el experimento "
                f"con {cores} cores y {percentage}% de datos"
            )


print("Scale-Up terminado.\n")

===== SCALE-UP =====
Repetición 1/1 | cores=1 | datos=10%
Repetición 1/1 | cores=2 | datos=20%
Repetición 1/1 | cores=4 | datos=40%
Repetición 1/1 | cores=8 | datos=80%
Repetición 1/1 | cores=10 | datos=100%
Scale-Up terminado.



In [6]:
import subprocess

max_depth = 10

resultado = subprocess.run(
    [
        "python",
        "plot-scalability-global.py",
        str(max_depth)
    ],
    capture_output=True,
    text=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERROR:")
    print(resultado.stderr)